# 🎙️ Whisper Transcription — Advanced Gradio Colab App (FoziScribe-style)

Ye advanced version **faster-whisper** (CTranslate2) use karta hai jo original `openai-whisper` se **4-5x zyada fast** aur **kam VRAM** leta hai, saath hi built-in **VAD (Voice Activity Detection)** hai jo asli silence detect karta hai — sirf word-gap guess nahi.

### Kya naya hai is version mein:
- ⚡ **Fast** aur 🎯 **Accurate** mode (bilkul FoziScribe jaisa — 1x vs 2x processing)
- 🔇 **Real VAD-based silence detection** (Silero VAD built-in), sirf timestamp-gap nahi
- 🚫 **Hallucination/repetition fix** (`condition_on_previous_text=False` + temperature fallback ladder)
- 📊 **Per-line confidence score** — low-confidence lines ⚠️ se flag hote hain
- 📁 4 export formats: **TXT, SRT, VTT, JSON** (word-level timestamps ke saath)
- 🌍 Language auto-detect confidence % ke saath dikhta hai

### Zaroori: Pehle GPU on karein
**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

## Step 1: Dependencies install karein

In [ ]:
!pip install -q faster-whisper gradio
!apt-get -qq install -y ffmpeg

## Step 2: Models lazy-load karein

Dono models (Fast + Accurate) ek saath load nahi karte — jab pehli baar us mode ka use ho, tabhi load hota hai. Isse VRAM bachta hai aur Colab T4 (15GB) per dono comfortably fit ho jate hain.

In [ ]:
from faster_whisper import WhisperModel
import torch

# "Fast" mode = distil-large-v3 (halka, tez, thodi kam accurate)
# "Accurate" mode = large-v3 (poora model, zyada accurate, thoda slow)
MODEL_CONFIGS = {
    "Fast": {"name": "distil-large-v3"},
    "Accurate": {"name": "large-v3"},
}

_loaded_models = {}

def get_model(mode):
    if mode not in _loaded_models:
        cfg = MODEL_CONFIGS[mode]
        device = "cuda" if torch.cuda.is_available() else "cpu"
        compute_type = "float16" if device == "cuda" else "int8"
        print(f"Loading {cfg['name']} ({mode} mode) on {device} [{compute_type}]...")
        _loaded_models[mode] = WhisperModel(cfg["name"], device=device, compute_type=compute_type)
        print("Loaded.")
    return _loaded_models[mode]

# Optional: warm up Fast mode now so the first transcription isn't slow
get_model("Fast")
print("Ready.")

## Step 3: Pause-based segmentation + de-dup + confidence scoring

Is version mein 3 fixes add kiye gaye hain jo real output mein dekhi gayi problems solve karte hain:

1. **Duplicate-word remover** — jab VAD chunk boundary per ya hallucination se koi word turant repeat ho (jaise `down. down.`, `something something`, `the The`), wo automatically detect ho kar hata diya jata hai (case/punctuation-insensitive comparison, ~1 second window).
2. **Hard max_chars enforcement** — pehle line `max_chars` se thodi zyada ho sakti thi (ek extra word tak) agar punctuation/pause na mile. Ab line ko **exceed hone se pehle hi break** kiya jata hai — koi bhi line kabhi limit cross nahi karegi.
3. **VAD tuning** — `speech_pad_ms` kam kiya aur `min_silence_duration_ms` badhaya taake chunk boundaries kam bane, jisse boundary-duplicate artifacts kam hon.

In [ ]:
import re

def format_mmss(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

def srt_timestamp(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = round((seconds - int(seconds)) * 1000)
    if ms == 1000:
        ms = 0
        s += 1
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def vtt_timestamp(seconds):
    return srt_timestamp(seconds).replace(",", ".")


def _normalize_word(text):
    """Lowercase + strip punctuation, for duplicate comparison only (not for display)."""
    return re.sub(r"[^\w']", "", text).lower()


def dedupe_words(all_words, max_gap=1.0):
    """Removes immediately-repeated words caused by VAD chunk-boundary re-transcription
    or model stutter/hallucination (e.g. 'down. down.', 'something something', 'the The').
    Keeps the FIRST occurrence, drops the repeat if it follows within max_gap seconds.
    """
    cleaned = []
    for w in all_words:
        if cleaned:
            prev = cleaned[-1]
            same_word = _normalize_word(prev["word"]) == _normalize_word(w["word"])
            close_in_time = (w["start"] - prev["end"]) <= max_gap
            if same_word and _normalize_word(w["word"]) != "" and close_in_time:
                continue  # drop this duplicate, keep prev
        cleaned.append(w)
    return cleaned


def filter_low_confidence_orphans(all_words, prob_threshold=0.20):
    """Drops isolated single words with very low confidence (e.g. 3%, 13%) that
    aren't exact duplicates but are still VAD-boundary / misheard-word artifacts
    (e.g. 'smoking?' -> 'poking?', or a stray word between two clean segments).
    Only removes words BELOW the threshold — real, clearly-spoken words almost
    always score well above this, so genuine speech is not affected.
    """
    return [w for w in all_words if w.get("prob", 1.0) >= prob_threshold]


def _finalize_line(current_words):
    text = "".join(x["word"] for x in current_words).strip()
    confidences = [x.get("prob", 1.0) for x in current_words]
    avg_conf = sum(confidences) / len(confidences) if confidences else 1.0
    return {
        "text": text,
        "start": current_words[0]["start"],
        "end": current_words[-1]["end"],
        "confidence": avg_conf,
    }


def build_pause_based_lines(all_words, pause_threshold=0.35, max_chars=50):
    """all_words: list of dicts with 'word', 'start', 'end', 'prob' (already de-duped)"""
    lines = []
    current_words = []

    for i, w in enumerate(all_words):
        # Hard cap: if adding this word would push the line over max_chars, close the
        # current line FIRST (so no line ever exceeds the limit), then start a new one.
        projected_text = ("".join(x["word"] for x in current_words) + w["word"]).strip()
        if current_words and len(projected_text) > max_chars:
            lines.append(_finalize_line(current_words))
            current_words = []

        current_words.append(w)
        current_text = "".join(x["word"] for x in current_words).strip()

        is_last_word = (i == len(all_words) - 1)
        ends_with_punct = w["word"].strip().endswith((".", ",", "?", "!"))

        gap_to_next = 0
        if not is_last_word:
            gap_to_next = all_words[i + 1]["start"] - w["end"]

        should_break = (
            is_last_word
            or ends_with_punct
            or gap_to_next > pause_threshold
            or len(current_text) >= max_chars
        )

        if should_break and current_text:
            lines.append(_finalize_line(current_words))
            current_words = []

    return lines

## Step 4: Transcription function (VAD + dual mode) aur Gradio interface

In [ ]:
import gradio as gr
import os
import json as json_lib

def transcribe_audio(file_path, language, mode, pause_threshold, max_chars, show_confidence, min_confidence):
    if file_path is None:
        return "Pehle koi file upload karein.", None, None, None, None

    lang = None if language == "Auto-detect" else language
    model = get_model(mode)

    # Accurate mode = wider beam search, Fast mode = greedy-ish
    beam_size = 5 if mode == "Accurate" else 1
    best_of = 5 if mode == "Accurate" else 1

    segments, info = model.transcribe(
        file_path,
        language=lang,
        word_timestamps=True,
        beam_size=beam_size,
        best_of=best_of,
        vad_filter=True,  # real silence detection, filters out noise/dead air
        # Fewer, longer chunks = fewer boundaries = fewer duplicate-word artifacts.
        # Less padding = less chance of the same audio being transcribed twice.
        vad_parameters=dict(min_silence_duration_ms=500, speech_pad_ms=100),
        condition_on_previous_text=False,  # avoids repetition-loop hallucinations
        temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],  # fallback ladder on low-confidence segments
    )

    raw_words = []
    for seg in segments:
        if seg.words:
            for w in seg.words:
                raw_words.append({
                    "word": w.word,
                    "start": w.start,
                    "end": w.end,
                    "prob": w.probability,
                })

    if not raw_words:
        return "Koi speech detect nahi hui — audio check karein.", None, None, None, None

    # Remove VAD-boundary / hallucination duplicate words before building lines
    all_words = dedupe_words(raw_words, max_gap=1.0)
    # Remove isolated low-confidence artifact words (e.g. misheard "smoking" -> "poking")
    all_words = filter_low_confidence_orphans(all_words, prob_threshold=min_confidence)

    if not all_words:
        return "Sirf low-confidence artifacts mile — 'Low-confidence filter' threshold kam karein.", None, None, None, None

    lines = build_pause_based_lines(all_words, pause_threshold=pause_threshold, max_chars=int(max_chars))

    display_lines = []
    for l in lines:
        prefix = f"[{format_mmss(l['start'])}]"
        if show_confidence:
            conf_pct = int(l["confidence"] * 100)
            flag = " ⚠️" if l["confidence"] < 0.6 else ""
            display_lines.append(f"{prefix} ({conf_pct}%){flag} {l['text']}")
        else:
            display_lines.append(f"{prefix} {l['text']}")

    header = f"Detected language: {info.language} ({info.language_probability:.0%} confidence)\nMode: {mode}\n\n"
    display_text = header + "\n".join(display_lines)

    base_name = os.path.splitext(os.path.basename(file_path))[0]

    # --- TXT export ---
    txt_path = f"/content/{base_name}_transcript.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(display_text)

    # --- SRT export ---
    srt_path = f"/content/{base_name}.srt"
    with open(srt_path, "w", encoding="utf-8") as f:
        for i, l in enumerate(lines, start=1):
            f.write(f"{i}\n{srt_timestamp(l['start'])} --> {srt_timestamp(l['end'])}\n{l['text']}\n\n")

    # --- VTT export ---
    vtt_path = f"/content/{base_name}.vtt"
    with open(vtt_path, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")
        for l in lines:
            f.write(f"{vtt_timestamp(l['start'])} --> {vtt_timestamp(l['end'])}\n{l['text']}\n\n")

    # --- JSON export (word-level, for downstream tools / video editors) ---
    json_path = f"/content/{base_name}_words.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json_lib.dump({
            "language": info.language,
            "language_probability": info.language_probability,
            "mode": mode,
            "lines": lines,
            "words": all_words,
        }, f, ensure_ascii=False, indent=2)

    return display_text, txt_path, srt_path, vtt_path, json_path


languages = ["Auto-detect", "en", "ur", "hi", "ar", "fr", "es", "de", "zh", "ja"]

with gr.Blocks(title="Whisper Transcription Tool — Advanced") as demo:
    gr.Markdown("# 🎙️ Audio/Video Transcription (Advanced)\nFile upload karein — Fast ya Accurate mode choose karein, VAD-based accurate pause timestamps hasil karein.")
    with gr.Row():
        with gr.Column():
            audio_input = gr.File(label="Audio/Video file upload karein", type="filepath")
            mode_dropdown = gr.Radio(["Fast", "Accurate"], value="Fast", label="Mode — Fast = distil-large-v3 (tez), Accurate = large-v3 (behtar accuracy)")
            lang_dropdown = gr.Dropdown(languages, value="Auto-detect", label="Language")
            pause_slider = gr.Slider(0.1, 1.0, value=0.35, step=0.05, label="Pause sensitivity (seconds) — kam value = zyada breaks")
            max_chars_slider = gr.Slider(10, 300, value=50, step=5, label="Max characters per line — flexible: chahain to 20 rakhein, chahain to 200+ (jitna chota, utni zyada chhoti/frequent lines)")
            confidence_checkbox = gr.Checkbox(value=True, label="Show per-line confidence (low-confidence lines ⚠️ se flag hongi)")
            min_confidence_slider = gr.Slider(0.0, 0.5, value=0.20, step=0.05, label="Low-confidence word filter — isse kam confidence wale orphan/misheard words hatayein (0 = filter off)")
            submit_btn = gr.Button("Transcribe karein", variant="primary")
        with gr.Column():
            output_text = gr.Textbox(label="Transcript (pause-based timestamps)", lines=20)
            txt_file = gr.File(label="TXT file download karein")
            srt_file = gr.File(label="SRT (subtitle) file download karein")
            vtt_file = gr.File(label="VTT (web subtitle) file download karein")
            json_file = gr.File(label="JSON (word-level timestamps) download karein")

    submit_btn.click(
        fn=transcribe_audio,
        inputs=[audio_input, lang_dropdown, mode_dropdown, pause_slider, max_chars_slider, confidence_checkbox, min_confidence_slider],
        outputs=[output_text, txt_file, srt_file, vtt_file, json_file]
    )

demo.launch(share=True, debug=True)

---
### Notes — kya behtar hua FoziScribe ke muqable + latest fixes

- **VAD (Voice Activity Detection)**: `vad_filter=True` asli silence/noise detect karta hai audio waveform se — sirf word-timestamp gaps guess nahi karta.
- **Duplicate-word fix**: `dedupe_words()` un artifacts ko hata deta hai jahan VAD chunk-boundary per ya model-stutter se koi word turant repeat ho jata tha (jaise `down. down.`, `something something else`, `the The`).
- **Low-confidence orphan filter (naya)**: `filter_low_confidence_orphans()` un isolated words ko hata deta hai jo exact duplicate nahi hain lekin bohot low-confidence misheard artifacts hain (jaise `"smoking?"` ko `"poking?"` sun lena, 3% confidence per). UI mein ek naya slider hai — "Low-confidence word filter" (default 0.20) — jise aap zaroorat ke hisaab se kam/zyada kar sakte hain.
- **Hard max_chars enforcement**: line ab **kabhi bhi** set kiye gaye `max_chars` se zyada lambi nahi hogi — chahe Whisper punctuation predict na kare ya koi pause na mile, line ko limit per pohanchte hi force-break kiya jata hai.
- **VAD tuning**: `min_silence_duration_ms=500` aur `speech_pad_ms=100` — kam boundaries, kam overlap, kam duplicate artifacts.
- **Hallucination fix**: `condition_on_previous_text=False` + temperature fallback ladder.
- **Fast vs Accurate**: `distil-large-v3` (~5x tez) vs `large-v3` (zyada accurate).
- **Confidence flagging**: har line ka average word-probability dikhta hai; 60% se kam wali lines ⚠️ se flag hoti hain.
- **Flexible max_chars slider**: 10 se 300 tak.
- **4 export formats**: TXT, SRT, VTT, aur word-level JSON.

### Tuning tips
- Agar genuine words filter ho rahe hain (bohot achi audio quality ke bawajood), **"Low-confidence word filter" slider ko 0 per le aayein** (filter off).
- Agar phir bhi kuch artifact words reh jayein, slider ko 0.25–0.30 tak badha dein.
- Agar koi genuine intentional word-repeat miss ho jaye (jaise koi jaan-boojh kar "no, no, no!" bole), `dedupe_words()` mein `max_gap=1.0` ko `0.4` kar dein.